In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('../data/processed/land_dataset_final_v3_1_keep.csv')

In [5]:
df = df.drop_duplicates()

In [8]:
df

,address_subdivision,address_locality,address_line_2,price_per_m2,land_area,price,longitude,latitude,near_Koh_Pich_in_km,Koh_Pich_nearest,...,f_unclassified,f_unused,h_id_price_mean,h_id_price_max,h_id_price_median,h_id_price_min,geometry,index_right,population,h_id
0,Phnom Penh,Mean Chey,Stueng Mean Chey,3357.98,52,174614.96,104.883100,11.552932,6,0,...,0,0,3231.734444,3443.50,3312.145,2879.39,POINT (104.8831002206026 11.55293201259622),53933,15646.0,8865846a91fffff
2,Phnom Penh,Chamkar Mon,Phsar Daeum Thkov,3809.30,178,678055.40,104.915003,11.528833,3,0,...,0,0,4163.095641,5124.69,4090.020,2719.42,POINT (104.9150031709444 11.52883307617736),53907,27484.0,8865846acbfffff
5,Phnom Penh,Saensokh,Phnom Penh Thmei,3437.03,138,474310.14,104.886163,11.586713,7,0,...,0,0,3936.791837,4822.66,3994.090,3166.66,POINT (104.8861626524092 11.58671276688701),54092,4251.0,88658468cbfffff
8,Phnom Penh,Saensokh,Phnom Penh Thmei,3469.70,162,562091.40,104.889529,11.575790,6,0,...,0,0,3607.445238,4360.25,3553.255,3075.99,POINT (104.8895286363168 11.57578950809717),53919,5039.0,8865846ab1fffff
12,Phnom Penh,Doun Penh,Chakto Mukh,5442.34,200,1088468.00,104.958218,11.558388,1,0,...,0,0,4536.392727,5881.00,4842.500,1880.12,POINT (104.958218177304 11.55838750186468),53963,8.0,8865846a39fffff
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23698,Phnom Penh,Chraoy Chongvar,Bak Kaeng,729.57,182,132781.74,104.929226,11.701893,16,0,...,0,0,673.093333,738.42,697.845,557.25,POINT (104.9292263210374 11.70189325769832),38894,1681.0,886586a699fffff
23699,Phnom Penh,Chraoy Chongvar,Preaek Ta Sek,571.98,212,121259.76,104.899855,11.667508,13,0,...,0,0,1243.465000,1875.82,1585.710,570.28,POINT (104.8998547375026 11.66750778225148),54026,4.0,8865846995fffff
23701,Phnom Penh,Praek Pnov,Ponsang,260.40,134,34893.60,104.756877,11.633307,22,0,...,0,0,221.971111,260.40,205.870,190.72,POINT (104.7568767794476 11.63330739755811),53735,599.0,8865846d85fffff
23702,Phnom Penh,Pur SenChey,Kantaok,1093.83,230,251580.90,104.785133,11.523526,17,0,...,0,0,1110.019333,1318.26,1124.820,892.05,POINT (104.7851334702664 11.52352649547996),53669,762.0,8865846e31fffff


In [6]:
df.shape

(9272, 240)

In [ ]:
# df.to_csv('../data/processed/land_dataset_final_v3_1_keep.csv', index=False)

In [13]:
import pandas as pd
from math import radians, sin, cos, sqrt, atan2

def haversine(lon1, lat1, lon2, lat2):
    """
    Calculate the great circle distance between two points 
    on the earth (specified in decimal degrees)
    """
    # Convert decimal degrees to radians 
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])

    # Haversine formula 
    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a)) 
    r = 6371  # Radius of earth in kilometers
    return c * r * 1000  # Return distance in meters

def reduce_density(df, min_distance_meters=50):
    """
    Reduce property density by removing properties that are too close to each other.
    
    Args:
        df (pd.DataFrame): DataFrame containing property data with 'latitude' and 'longitude' columns
        min_distance_meters (int): Minimum allowed distance between properties in meters
        
    Returns:
        pd.DataFrame: Reduced DataFrame where no two properties are closer than min_distance_meters
    """
    if len(df) == 0:
        return df
    
    # Make a copy to avoid modifying original
    reduced_df = df.copy()
    
    # Convert to list of tuples for processing
    properties = list(reduced_df[['latitude', 'longitude']].itertuples(index=False, name=None))
    indices = list(reduced_df.index)
    
    # Initialize variables
    removed_indices = set()
    changed = True
    
    # Keep processing until no more changes are needed
    while changed:
        changed = False
        n = len(properties)
        
        # Compare each pair of properties
        for i in range(n):
            if i in removed_indices:
                continue
                
            lat1, lon1 = properties[i]
            
            for j in range(i+1, n):
                if j in removed_indices:
                    continue
                    
                lat2, lon2 = properties[j]
                
                # Calculate distance in meters
                distance = haversine(lon1, lat1, lon2, lat2)
                
                if distance < min_distance_meters:
                    # Remove the property with higher index (could implement other criteria)
                    removed_indices.add(j)
                    changed = True
    
    # Create the reduced DataFrame
    keep_indices = [idx for i, idx in enumerate(indices) if i not in removed_indices]
    result_df = reduced_df.loc[keep_indices].reset_index(drop=True)
    
    # Verify no properties are too close in final result
    final_coords = list(result_df[['latitude', 'longitude']].itertuples(index=False, name=None))
    for i in range(len(final_coords)):
        for j in range(i+1, len(final_coords)):
            lat1, lon1 = final_coords[i]
            lat2, lon2 = final_coords[j]
            distance = haversine(lon1, lat1, lon2, lat2)
            if distance < min_distance_meters:
                print(f"Warning: Found properties {distance:.2f}m apart in final result")
    
    print(f"Reduced from {len(df)} to {len(result_df)} properties "
          f"(removed {len(df)-len(result_df)} properties)")
    
    return result_df

# Example usage:
reduced_properties = reduce_density(df, min_distance_meters=150)
# reduced_properties.to_csv('reduced_properties.csv', index=False)

Reduced from 9272 to 5493 properties (removed 3779 properties)


In [15]:
reduced_properties.to_csv('../data/processed/land_dataset_final_v3_1_keep_1.csv', index=False)